# HKI Threat Catalog — All 15 Attacks, Before and After

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/04_threat_demos.ipynb)

Each cell pair shows:
- **VULNERABLE** — the code pattern found in production agentic systems today
- **FIXED** — the same code with HKI enforcement added

All demos are hermetic: no LLM API key, no network calls, no infrastructure required.

| # | Threat | Category | Fix primitive |
|---|--------|----------|---------------|
| T01 | Semantic cache cross-domain leak | Cache | `derive_hki_cache_key` |
| T02 | Body-parameter scope override | Scope | `reject_conflicting_scope_argument` |
| T03 | Implicit `or 'global'` fallback | Scope | `is_forbidden_runtime_domain` |
| T04 | Async job loses domain on resume | Async | Envelope in job payload |
| T05 | Vector index shared without filter | Retrieval | `assert_artifact_visible` |
| T06 | MCP tool without domain binding | Tool | `evaluate_gateway_target` |
| T07 | A2A delegation drops envelope | Agent | `validate_envelope` on receive |
| T08 | Embedding cache omits domain | Cache | `derive_hki_cache_key` |
| T09 | Admin route reachable at runtime | Plane | `purpose` + `risk_tier` gate |
| T10 | Wildcard publication | Scope | `is_forbidden_runtime_domain` |
| T11 | Envelope replay | Integrity | Nonce store |
| T12 | Expired envelope accepted | Integrity | `validate_envelope` |
| T13 | Version downgrade | Integrity | `validate_envelope` |
| T14 | Graph traversal crosses domain edges | Retrieval | `same_domain` on every edge |
| T15 | Prompt-injected scope override | LLM | `reject_conflicting_scope_argument` |

**Time to run:** ~3 minutes (run all cells sequentially)

In [ ]:
%pip install hki-runtime -q

In [ ]:
import time
import hki_runtime

# Shared envelope factory used throughout this notebook
def envelope(domain: str, purpose: str = "retrieve", expires_offset: int = 99_999_999_999) -> dict:
    return {
        "hki_version": "1.0",
        "envelope_id": f"env_{domain}_{int(time.time())}",
        "org_id": "org_acme",
        "subject_id": "user_42",
        "active_domain": domain,
        "authorized_domains": [domain],
        "purpose": purpose,
        "risk_tier": "read-only",
        "policy_pack_id": f"{domain}@2026-05",
        "issued_at": 0,
        "expires_at": expires_offset,
        "issuer": "gateway.acme.internal",
        "signature": "ed25519:placeholder",
    }

def show(label, result, leaked=True):
    icon = "🔴 LEAK" if leaked else "🟢 BLOCKED"
    print(f"{icon}  {label}: {result}")

print("Setup complete")

---
## Category 1 — Cache Leaks

Cache keys that don't include the domain allow one tenant's cached responses
to be served to a different tenant asking the same question.

### T01 — Semantic cache cross-domain leak

**Real-world scenario:** A pharmacy agent asks "what is the return policy?"
The answer is cached. A travel agent asks the same question — and receives
the pharmacy answer from cache because the key is just the query string.

**Root cause:** Cache key = `query`. Missing: `org_id`, `domain`, `model_route`.

In [ ]:
# VULNERABLE
CACHE = {}

def respond_buggy(query: str, domain: str) -> str:
    if query in CACHE:        # BUG: no domain in key
        return CACHE[query]
    answer = f"[{domain}] {query}"
    CACHE[query] = answer
    return answer

pharmacy = respond_buggy("return policy", "pharmacy")
travel   = respond_buggy("return policy", "travel")    # hits the cache

show("travel got pharmacy answer", travel, leaked=True)
assert pharmacy == travel

In [ ]:
# FIXED — derive_hki_cache_key includes org + domain + operation
SAFE_CACHE = {}

def respond_safe(query: str, domain: str) -> str:
    key = hki_runtime.derive_hki_cache_key({
        "envelope": envelope(domain),
        "operation": "chat.completion",
        "input": {"query": query},
    })
    if key in SAFE_CACHE:
        return SAFE_CACHE[key]
    answer = f"[{domain}] {query}"
    SAFE_CACHE[key] = answer
    return answer

pharmacy = respond_safe("return policy", "pharmacy")
travel   = respond_safe("return policy", "travel")

show("domains get separate cache entries", pharmacy != travel, leaked=False)
print(f"  pharmacy: {pharmacy}")
print(f"  travel:   {travel}")

### T08 — Embedding cache omits domain

**Real-world scenario:** Two tenants embed the same phrase. Tenant B receives
tenant A's cached embedding vector — which was computed with tenant A's
fine-tuned tokenizer or custom model, making similarity search nonsensical and
potentially exposing tenant-specific semantic space.

**Root cause:** Embedding cache key = `model:text`. Missing: `org_id`, `domain`.

In [ ]:
# VULNERABLE
EMBED_CACHE = {}

def embed_buggy(text: str, model: str, domain: str) -> list:
    key = f"{model}:{text}"     # BUG: no domain
    if key in EMBED_CACHE:
        return EMBED_CACHE[key]
    vec = [float(len(text)), float(hash(domain) & 0xFF)]  # domain-specific
    EMBED_CACHE[key] = vec
    return vec

iris_vec  = embed_buggy("hello", "text-embedding-3", "iris")
pulse_vec = embed_buggy("hello", "text-embedding-3", "pulse")  # hits cache

show("pulse got iris embedding", f"iris={iris_vec} pulse={pulse_vec}", leaked=True)
assert iris_vec == pulse_vec

In [ ]:
# FIXED
SAFE_EMBED_CACHE = {}

def embed_safe(text: str, model: str, domain: str) -> list:
    key = hki_runtime.derive_hki_cache_key({
        "envelope": envelope(domain),
        "operation": "embedding.compute",
        "input": {"text": text},
        "model_route": model,
    })
    if key in SAFE_EMBED_CACHE:
        return SAFE_EMBED_CACHE[key]
    vec = [float(len(text)), float(hash(domain) & 0xFF)]
    SAFE_EMBED_CACHE[key] = vec
    return vec

iris_vec  = embed_safe("hello", "text-embedding-3", "iris")
pulse_vec = embed_safe("hello", "text-embedding-3", "pulse")

show("separate embedding cache per domain", iris_vec != pulse_vec, leaked=False)
print(f"  iris:  {iris_vec}")
print(f"  pulse: {pulse_vec}")

---
## Category 2 — Scope Override

Code that trusts a domain name from an untrusted source (request body, query string,
LLM output) instead of the signed envelope.

### T02 — Body-parameter scope override

**Real-world scenario:** An agent sends `{"scope": "pulse"}` in the request body.
The retrieval service reads the scope from the body and ignores the signed envelope.

In [ ]:
# VULNERABLE
DATA = {"iris": ["iris-doc-1", "iris-doc-2"], "pulse": ["pulse-doc-1"]}

def search_buggy(env: dict, body: dict) -> list:
    domain = body.get("scope") or env["active_domain"]  # BUG: body wins
    return DATA.get(domain, [])

env = envelope("iris")
leaked = search_buggy(env, {"scope": "pulse"})  # override the envelope
show("iris envelope read pulse data", leaked, leaked=True)

In [ ]:
# FIXED
def search_safe(env: dict, body: dict) -> list:
    err = hki_runtime.reject_conflicting_scope_argument(env, body)
    if err:
        raise PermissionError(err)
    return DATA.get(env["active_domain"], [])

try:
    search_safe(envelope("iris"), {"scope": "pulse"})
except PermissionError as e:
    show("body scope override", str(e), leaked=False)

### T03 — Implicit `or 'global'` fallback

**Real-world scenario:** A context object is passed through a chain of services.
One service loses the domain. The retrieval code falls back to `"global"` which,
in most implementations, means "no filter" — returning every row in the index.

In [ ]:
# VULNERABLE — this exact pattern is in thousands of production RAG pipelines
ALL_DATA = {"iris": ["iris-1"], "pulse": ["pulse-1"]}

def query_buggy(ctx: dict) -> list:
    domain = ctx.get("active_domain") or "global"   # BUG: 'global' = no filter
    if domain == "global":
        return [doc for rows in ALL_DATA.values() for doc in rows]
    return ALL_DATA.get(domain, [])

leaked = query_buggy({})   # empty context — domain was dropped somewhere upstream
show("empty context returned all rows", leaked, leaked=True)

In [ ]:
# FIXED
def query_safe(ctx: dict) -> list:
    domain = ctx.get("active_domain")
    if not domain or hki_runtime.is_forbidden_runtime_domain(domain):
        raise PermissionError("active_domain required and must not be global/wildcard")
    return ALL_DATA.get(domain, [])

try:
    query_safe({})
except PermissionError as e:
    show("empty context", str(e), leaked=False)

try:
    query_safe({"active_domain": "global"})
except PermissionError as e:
    show("global domain", str(e), leaked=False)

### T10 — Wildcard publication

**Real-world scenario:** A document is published with `domain="*"`. Every
subsequent runtime read returns it regardless of active domain — effectively
a global injection via the content layer.

In [ ]:
# VULNERABLE
STORE = []

def publish_buggy(content, domain):
    STORE.append({"domain": domain, "content": content})

def read_buggy(domain):
    return [a for a in STORE if a["domain"] in (domain, "*", "global")]

publish_buggy("backdoor content", domain="*")   # wildcard publish
visible = read_buggy("iris")
show("iris sees wildcard-published doc", visible, leaked=True)

In [ ]:
# FIXED
SAFE_STORE = []

def publish_safe(content, domain):
    if hki_runtime.is_forbidden_runtime_domain(domain):
        raise PermissionError(f"wildcard/global publication refused: {domain!r}")
    SAFE_STORE.append({"domain": domain, "content": content})

def read_safe(domain):
    if hki_runtime.is_forbidden_runtime_domain(domain):
        raise PermissionError(f"runtime read refused for {domain!r}")
    return [a for a in SAFE_STORE if hki_runtime.same_domain(a["domain"], domain)]

try:
    publish_safe("backdoor content", domain="*")
except PermissionError as e:
    show("wildcard publication", str(e), leaked=False)

### T15 — Prompt-injected scope override

**Real-world scenario:** An LLM is instructed (via injected text in a document)
to call a tool with `domain="pulse"`. The tool trusts the LLM-supplied argument
rather than the signed envelope.

**Why this matters:** Even a read-only RAG agent can be weaponised to exfiltrate
cross-domain data if tool arguments bypass the envelope.

In [ ]:
# VULNERABLE — tool accepts domain from LLM-generated arguments
SECRETS = {"iris": "iris-public", "pulse": "pulse-secret"}

def search_tool_buggy(query: str, domain: str, env: dict) -> str:
    return SECRETS.get(domain, "")   # BUG: uses LLM arg, not envelope

env = envelope("iris")
# LLM was injected to call with domain="pulse"
leaked = search_tool_buggy("anything", domain="pulse", env=env)
show("prompt-injected call read pulse secret", leaked, leaked=True)

In [ ]:
# FIXED — envelope wins; conflicting LLM arg is rejected
def search_tool_safe(query: str, domain: str, env: dict) -> str:
    err = hki_runtime.reject_conflicting_scope_argument(env, {"domain": domain})
    if err:
        raise PermissionError(err)
    return SECRETS.get(env["active_domain"], "")

try:
    search_tool_safe("anything", domain="pulse", env=envelope("iris"))
except PermissionError as e:
    show("prompt-injected domain arg", str(e), leaked=False)

---
## Category 3 — Retrieval Leaks

Retrieval code that fetches documents or graph nodes across domain boundaries.

### T05 — Vector index shared without per-row domain filter

**Real-world scenario:** All tenants write to a shared vector index.
A similarity search forgets the `WHERE domain = ?` filter — and returns
the closest vectors regardless of tenant.

In [ ]:
# VULNERABLE
INDEX = [
    {"id": "v1", "domain": "iris",  "text": "iris-only prescription data"},
    {"id": "v2", "domain": "pulse", "text": "pulse-only travel booking data"},
    {"id": "v3", "domain": "iris",  "text": "iris general doc"},
]

def search_buggy(query: str, domain: str) -> list:
    return [r for r in INDEX if query in r["text"]]   # BUG: no domain filter

results = search_buggy("data", "iris")
leaked = [r for r in results if r["domain"] != "iris"]
show("iris query returned pulse rows", leaked, leaked=True)

In [ ]:
# FIXED — assert_artifact_visible on every row
def search_safe(query: str, domain: str) -> list:
    env = envelope(domain)
    candidates = [r for r in INDEX if query in r["text"]]
    return [
        r for r in candidates
        if hki_runtime.assert_artifact_visible(env, {
            "org_id": "org_acme",
            "domain": r["domain"],
            "artifact_type": "vector",
            "artifact_id": r["id"],
        }) is None
    ]

results = search_safe("data", "iris")
show("only iris rows returned", [r["id"] for r in results], leaked=False)
assert all(r["domain"] == "iris" for r in results)

### T14 — Graph traversal crosses domain edges

**Real-world scenario:** A knowledge graph has nodes from multiple domains.
A traversal correctly filters the *starting* node by domain, but follows
edges to neighboring nodes without re-checking their domain.
One cross-domain edge exposes the entire connected component.

In [ ]:
# VULNERABLE
NODES = {
    "n1": {"domain": "iris",  "content": "iris home node"},
    "n2": {"domain": "pulse", "content": "pulse secret node"},  # different domain
}
EDGES = [("n1", "n2")]   # cross-domain edge exists in graph

def traverse_buggy(start: str, domain: str) -> list:
    if NODES[start]["domain"] != domain:
        return []
    # BUG: checks start node but NOT the destination nodes
    return [NODES[dst] for src, dst in EDGES if src == start]

found = traverse_buggy("n1", "iris")
leaked = [n for n in found if n["domain"] != "iris"]
show("iris traversal reached pulse node", leaked, leaked=True)

In [ ]:
# FIXED — same_domain check on every destination node
def traverse_safe(start: str, domain: str) -> list:
    if NODES[start]["domain"] != domain:
        return []
    out = []
    for src, dst in EDGES:
        if src != start:
            continue
        node = NODES[dst]
        if hki_runtime.same_domain(node["domain"], domain):  # re-check on every hop
            out.append(node)
    return out

found = traverse_safe("n1", "iris")
show("cross-domain edge blocked", found or "(no nodes — pulse node filtered)", leaked=False)
assert all(n["domain"] == "iris" for n in found)

---
## Category 4 — Agent & Tool Leaks

Multi-agent systems and tool routers that lose the domain across process boundaries.

### T04 — Async job loses domain on resume

**Real-world scenario:** An ingestion job is enqueued with a domain context.
When a worker picks it up (possibly minutes later, possibly a different process),
the domain is gone — and the worker defaults to `"global"`, labelling the artifact
as accessible to everyone.

In [ ]:
# VULNERABLE
QUEUE, ARTIFACTS = [], []

def enqueue_buggy(payload: dict, domain: str):
    QUEUE.append({"payload": payload})   # BUG: domain not in message

def worker_buggy():
    for msg in QUEUE:
        ARTIFACTS.append({"domain": "global", "data": msg["payload"]})  # no domain

enqueue_buggy({"text": "iris prescription doc"}, domain="iris")
worker_buggy()
show("artifact stored as 'global'", ARTIFACTS[-1], leaked=True)

In [ ]:
# FIXED — envelope travels in the job message; worker re-validates
SAFE_QUEUE, SAFE_ARTIFACTS = [], []

def enqueue_safe(payload: dict, env: dict):
    SAFE_QUEUE.append({"payload": payload, "envelope": env})  # envelope in message

def worker_safe():
    for msg in SAFE_QUEUE:
        result = hki_runtime.validate_envelope(msg["envelope"])
        if not result.ok:
            raise PermissionError(f"job envelope invalid: {result.issues}")
        domain = result.envelope.active_domain
        SAFE_ARTIFACTS.append({"domain": domain, "data": msg["payload"]})

enqueue_safe({"text": "iris prescription doc"}, envelope("iris", purpose="ingest"))
worker_safe()
show("artifact stored with correct domain", SAFE_ARTIFACTS[-1]["domain"], leaked=False)

### T06 — MCP tool without domain binding

**Real-world scenario:** An MCP tool registry has tools with `domain="*"` or
missing domain labels. Any agent — regardless of active domain — can invoke them,
making them a privilege-escalation vector.

In [ ]:
# VULNERABLE
TOOLS = [
    {"id": "rx.lookup",     "domain": "iris"},
    {"id": "global.search", "domain": "*"},    # wildcard — reachable from everywhere
    {"id": "shared.tool"},                     # no domain — same problem
]

def can_invoke_buggy(tool_id: str, active_domain: str) -> bool:
    for t in TOOLS:
        if t["id"] == tool_id:
            d = t.get("domain", "*")
            return d in ("*", active_domain)   # BUG: wildcard passes
    return False

print(f"global.search from iris: {can_invoke_buggy('global.search', 'iris')}  ← should be False")
print(f"shared.tool   from iris: {can_invoke_buggy('shared.tool', 'iris')}  ← should be False")

In [ ]:
# FIXED — evaluate_gateway_target: missing or wildcard domain → denied
def can_invoke_safe(tool_id: str, domain: str) -> bool:
    tool = next((t for t in TOOLS if t["id"] == tool_id), None)
    if not tool:
        return False
    env = envelope(domain)
    tool_domain = tool.get("domain", "")  # missing domain → empty string → denied
    decision = hki_runtime.evaluate_gateway_target(env, {
        "type": "tool",
        "id": tool_id,
        "domain": tool_domain,
    })
    return decision.allowed

print(f"rx.lookup     from iris: {can_invoke_safe('rx.lookup', 'iris')}")
print(f"global.search from iris: {can_invoke_safe('global.search', 'iris')}  ← blocked")
print(f"shared.tool   from iris: {can_invoke_safe('shared.tool', 'iris')}  ← blocked")

### T07 — A2A delegation drops the envelope

**Real-world scenario:** Agent A (scoped to `iris`) delegates a subtask to Agent B.
The envelope is not forwarded. Agent B has no scope — it falls back to its service
identity which has access to all domains.

In [ ]:
# VULNERABLE
DOC_STORE = {"iris": ["iris-confidential"], "pulse": ["pulse-confidential"]}

def agent_b_buggy(task: str) -> list:
    return [d for docs in DOC_STORE.values() for d in docs]  # no scope → see all

def agent_a_delegate_buggy(task: str, caller_domain: str) -> list:
    return agent_b_buggy(task)   # BUG: envelope not forwarded

leaked = agent_a_delegate_buggy("find docs", caller_domain="iris")
show("iris delegation saw pulse docs", leaked, leaked=True)

In [ ]:
# FIXED — envelope passed and re-validated at each agent boundary
def agent_b_safe(task: str, env: dict) -> list:
    result = hki_runtime.validate_envelope(env, require_signature=True)
    if not result.ok:
        raise PermissionError(f"delegation envelope invalid: {result.issues}")
    return DOC_STORE.get(result.envelope.active_domain, [])

def agent_a_delegate_safe(task: str, env: dict) -> list:
    return agent_b_safe(task, env)   # envelope forwarded

result = agent_a_delegate_safe("find docs", envelope("iris"))
show("iris delegation sees only iris docs", result, leaked=False)
assert "pulse-confidential" not in result

---
## Category 5 — Envelope Integrity

Attacks that abuse the envelope itself: replaying it, using an expired one,
or dowgrading its version to bypass checks.

### T09 — Admin route reachable from runtime envelope

**Real-world scenario:** A runtime agent sends a `chat` request that somehow
reaches an admin route (delete index, reset policy). The route does not check
`purpose` or `risk_tier` — it only checks that an envelope is present.

In [ ]:
# VULNERABLE
INDEX = {"iris": ["doc-1", "doc-2"]}

def admin_delete_buggy(env: dict, domain: str) -> str:
    INDEX.pop(domain, None)   # BUG: no purpose/risk_tier check
    return f"deleted {domain}"

runtime_env = {"purpose": "chat", "risk_tier": "read-only", "active_domain": "iris"}
result = admin_delete_buggy(runtime_env, "iris")
show("chat envelope deleted an index", f"{result}, iris still exists: {'iris' in INDEX}", leaked=True)

In [ ]:
# FIXED — admin routes check purpose and risk_tier
INDEX = {"iris": ["doc-1", "doc-2"]}  # reset

def admin_delete_safe(env: dict, domain: str) -> str:
    if env.get("purpose") not in {"admin"} or env.get("risk_tier") not in {"admin", "destructive"}:
        raise PermissionError(
            f"runtime envelope (purpose={env.get('purpose')!r}) cannot reach admin route"
        )
    INDEX.pop(domain, None)
    return f"deleted {domain}"

try:
    admin_delete_safe({"purpose": "chat", "risk_tier": "read-only"}, "iris")
except PermissionError as e:
    show("runtime envelope at admin route", str(e), leaked=False)
    print(f"  iris index preserved: {'iris' in INDEX}")

### T11 — Envelope replay

**Real-world scenario:** An attacker captures a valid signed envelope
and replays it after the original request completed. Without a nonce store,
the same envelope is accepted indefinitely until it expires.

In [ ]:
# VULNERABLE
def handle_buggy(env: dict) -> str:
    return f"served {env['envelope_id']}"   # BUG: no nonce check

env = {"envelope_id": "env_abc", "active_domain": "iris"}
r1 = handle_buggy(env)
r2 = handle_buggy(env)   # replay
show("same envelope accepted twice", f"{r1!r} and {r2!r}", leaked=True)

In [ ]:
# FIXED — nonce store rejects seen envelope_ids
SEEN = set()

def handle_safe(env: dict) -> str:
    eid = env.get("envelope_id")
    if not eid:
        raise PermissionError("envelope_id missing")
    if eid in SEEN:
        raise PermissionError(f"envelope {eid!r} already consumed (replay blocked)")
    SEEN.add(eid)
    return f"served {eid}"

env = {"envelope_id": "env_abc", "active_domain": "iris"}
r1 = handle_safe(env)
print(f"First request:  {r1}")
try:
    handle_safe(env)   # replay attempt
except PermissionError as e:
    show("replay attempt", str(e), leaked=False)

### T12 — Expired envelope accepted

**Real-world scenario:** An envelope's `expires_at` is in the past. The validator
only checks the signature — not the clock. Stolen or leaked envelopes remain valid forever.

In [ ]:
# VULNERABLE
def accept_buggy(env: dict) -> bool:
    return bool(env.get("signature"))   # BUG: ignores expires_at

expired_env = {**envelope("iris"), "expires_at": 1}  # expired in 1970
show("expired envelope accepted", accept_buggy(expired_env), leaked=True)

In [ ]:
# FIXED — validate_envelope checks the clock
try:
    result = hki_runtime.validate_envelope(expired_env)
    if not result.ok:
        codes = [i.code for i in result.issues]
        show("expired envelope", f"rejected: {codes}", leaked=False)
    else:
        print("ERROR: expired envelope was accepted")
except Exception as e:
    show("expired envelope", str(e), leaked=False)

### T13 — Envelope version downgrade

**Real-world scenario:** An attacker forges an envelope with `hki_version: "0.9"`
hoping the validator skips checks that were added in version 1.0.

In [ ]:
# VULNERABLE
def accept_buggy(env: dict) -> bool:
    return bool(env.get("signature"))   # BUG: hki_version not checked

old_env = {"hki_version": "0.9", "signature": "sig", "active_domain": "iris"}
show("version 0.9 envelope accepted", accept_buggy(old_env), leaked=True)

In [ ]:
# FIXED
old_env_full = {**envelope("iris"), "hki_version": "0.9"}
result = hki_runtime.validate_envelope(old_env_full)
if not result.ok:
    codes = [i.code for i in result.issues]
    show("version 0.9 downgrade", f"rejected: {codes}", leaked=False)
else:
    print("ERROR: version downgrade was accepted")

---
## Summary — All 15 threats in one table

| ID | Threat | Fix primitive | One line |
|----|--------|--------------|----------|
| T01 | Semantic cache cross-domain leak | `derive_hki_cache_key` | Include `org_id + domain` in every cache key |
| T02 | Body-parameter scope override | `reject_conflicting_scope_argument` | Signed envelope wins over request body |
| T03 | Implicit `or 'global'` fallback | `is_forbidden_runtime_domain` | Fail closed — never fall back to global |
| T04 | Async job loses domain on resume | `validate_envelope` in worker | Envelope travels in the job message |
| T05 | Vector index without domain filter | `assert_artifact_visible` per row | Check every row, not just the query |
| T06 | MCP tool without domain binding | `evaluate_gateway_target` | Missing/wildcard domain → denied |
| T07 | A2A delegation drops envelope | `validate_envelope` on receive | Re-validate on every agent boundary |
| T08 | Embedding cache omits domain | `derive_hki_cache_key` | Same as T01 — domain in embedding key |
| T09 | Admin route reachable at runtime | `purpose` + `risk_tier` gate | Admin routes check both fields |
| T10 | Wildcard publication | `is_forbidden_runtime_domain` | Refuse `domain="*"` or `domain="global"` at write time |
| T11 | Envelope replay | nonce store (`SEEN` set) | One `envelope_id` = one request |
| T12 | Expired envelope accepted | `validate_envelope` | Check the clock, not just the signature |
| T13 | Version downgrade | `validate_envelope` | Only `hki_version: "1.0"` accepted |
| T14 | Graph traversal crosses domain edges | `same_domain` on every edge | Re-check on every hop, not just root |
| T15 | Prompt-injected scope override | `reject_conflicting_scope_argument` | LLM-supplied domain args cannot override envelope |

All fixes use five primitives from `hki-runtime`. If your stack enforces all five, you block all 15.

### Run the conformance suite against your own adapter

```bash
pip install hki-runtime
npx @hki/conformance ./your-adapter.js    # TypeScript
pytest packages/hki-conformance-py/       # Python (coming soon)
```

---
**Full threat catalog with runnable test files:** [docs/HKI_THREATS.md](../docs/HKI_THREATS.md)  
**Standard:** [spec/HKI-1.0.md](../spec/HKI-1.0.md)  
**GitHub:** https://github.com/h3nok/HKI